In [ ]:
import pandas as pd 
import numpy as np
from datetime import timedelta
import ta.volatility
data = pd.read_csv('../common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing.csv')
# Define parameters
dist1 = 14
dist2 = 21

# Calculate highest and lowest values over dist1 and dist2 periods
data['hhb1'] = data['high'].rolling(window=dist1, center=True).max()
data['llb1'] = data['low'].rolling(window=dist1, center=True).min()
data['hhb'] = data['high'].rolling(window=dist2, center=True).max()
data['llb'] = data['low'].rolling(window=dist2, center=True).min()

# Calculate ATR
data['atr'] = ta.volatility.average_true_range(data['high'], data['low'], data['close'], window=50)

# Initialize b1, b2, b3, b4
data['b1'] = np.nan
data['b2'] = np.nan
data['b3'] = np.nan
data['b4'] = np.nan

# Fill b1, b2, b3, b4 based on conditions
data.loc[data['high'] == data['hhb'], 'b1'] = data['high'] + data['atr']
data.loc[data['low'] == data['llb'], 'b2'] = data['low'] - data['atr']
data.loc[data['high'] == data['hhb1'], 'b3'] = data['high'] + data['atr'] / 2
data.loc[data['low'] == data['llb1'], 'b4'] = data['low'] - data['atr'] / 2

# Generate signals
conditions = [
    (data['b1'].notna() & data['b3'].notna(), "strong sell"),
    (data['b1'].notna() & data['b3'].isna(), "sell"),
    (data['b1'].isna() & data['b3'].notna(), "minor sell or exit buy"),
    (data['b2'].notna() & data['b4'].notna(), "strong buy"),
    (data['b2'].notna() & data['b4'].isna(), "buy"),
    (data['b2'].isna() & data['b4'].notna(), "minor buy or exit sell")
]

# Apply conditions to generate signals
for condition, signal in conditions:
    data.loc[condition, 'signal'] = signal
data['UTC'] = pd.to_datetime(data['datetime']) + timedelta(hours=5)
data['GMT'] = data['UTC'] + timedelta(hours=2)

# Save to CSV
output_file = '../common/MachineLearningModel/output/outputsupersignalV3Eurusd.csv'
data.to_csv(output_file, index=False)
# Display the DataFrame with buy and sell signals
# print(data[['timestamp', 'high', 'low', 'b1', 'b2', 'b3', 'b4']])